# Model 2 — Söz Yazarı (Lyrics Generator)
`{emotion, bpm, key, instruments, vocal_style}` → Türkçe şarkı sözü

Başlamadan önce:
1. Sağ panel → **Add Data** → **Upload** → `dataset.jsonl`
2. Accelerator: **GPU T4 x2**
3. Hücreleri sırayla çalıştır

In [ ]:
# ── 1. Kurulum ────────────────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece protobuf
print('Kurulum tamamlandi.')

In [ ]:
# ── 2. Dataset: Yükle + Lyrics Formatına Dönüştür ────────────────────────────
import glob, json
from datasets import Dataset

candidates = glob.glob('/kaggle/input/**/dataset.jsonl', recursive=True)
RAW_PATH = candidates[0] if candidates else '/kaggle/working/dataset.jsonl'
print(f'Ham dataset: {RAW_PATH}')

def fmt_instruments(v):
    return ', '.join(v) if isinstance(v, list) else str(v)

records = []
with open(RAW_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)
        lyrics = d.get('structured_lyrics', '').strip()
        # 20 kelimeden kisa sarkilari atla
        if not lyrics or len(lyrics.split()) < 20:
            continue
        lyrics_input = (
            f"Türkçe şarkı sözü yaz: "
            f"duygu={d.get('emotion','')}, "
            f"enerji={d.get('energy',5)}, "
            f"tempo={d.get('bpm',90)} BPM, "
            f"ton={d.get('key','A minor')}, "
            f"enstrümanlar={fmt_instruments(d.get('instruments',[]))}, "
            f"vokal={d.get('vocal_style','')}"
        )
        records.append({'input_text': lyrics_input, 'target_text': lyrics})

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=0.10, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Kayit: {len(records)} | Egitim: {len(train_ds)} | Dogrulama: {len(eval_ds)}')
print('Ornek girdi:', records[0]['input_text'])
print('Ornek cikti (ilk 150):', records[0]['target_text'][:150])

In [ ]:
# ── 3. Konfigürasyon ──────────────────────────────────────────────────────────
BASE_MODEL     = 'google/flan-t5-small'
OUTPUT_DIR     = '/kaggle/working/story-to-music-lyricist'
CHECKPOINT_DIR = '/kaggle/working/checkpoints-lyricist'

MAX_INPUT_LEN  = 64    # stil parametresi girdi kisadir
MAX_TARGET_LEN = 512   # sarki sozleri uzun olabilir

EPOCHS         = 25    # lyrics daha karmasik, daha fazla epoch
BATCH_SIZE     = 4     # uzun cikti nedeniyle kucuk batch
GRAD_ACCUM     = 4     # efektif batch = 16
LR             = 3e-4
WARMUP_RATIO   = 0.10
WEIGHT_DECAY   = 0.01
PATIENCE       = 5

print('Konfigurasyon tamam.')

In [ ]:
# ── 4. Model ve Tokenizer ─────────────────────────────────────────────────────
import torch
from transformers import T5ForConditionalGeneration, T5TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

tokenizer = T5TokenizerFast.from_pretrained(BASE_MODEL)
model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)
print(f'Parametre sayisi: {model.num_parameters():,}')

In [ ]:
# ── 5. Tokenizasyon ───────────────────────────────────────────────────────────
from transformers import DataCollatorForSeq2Seq

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)
print('Tokenizasyon tamam.')

In [ ]:
# ── 6. Eğitim ─────────────────────────────────────────────────────────────────
from pathlib import Path
import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

total_steps  = (len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=3,
    fp16=(device == 'cuda'),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=10,
    report_to='none',
    seed=42,
)

trainer_kwargs = dict(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=eval_tok,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)
ver = tuple(int(x) for x in transformers.__version__.split('.')[:2])
trainer_kwargs['processing_class' if ver >= (4, 46) else 'tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

checkpoints = sorted(Path(CHECKPOINT_DIR).glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1])) if Path(CHECKPOINT_DIR).exists() else []
last_ckpt = str(checkpoints[-1]) if checkpoints else None
if last_ckpt:
    print(f'Checkpoint bulundu: {last_ckpt}')

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# ── 7. Modeli Kaydet ──────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

total_mb = sum(f.stat().st_size for f in Path(OUTPUT_DIR).rglob('*') if f.is_file()) / (1024**2)
print(f'Model kaydedildi: {OUTPUT_DIR}  ({total_mb:.1f} MB)')

In [ ]:
# ── 8. Smoke Test ─────────────────────────────────────────────────────────────
ornek_input = 'Türkçe şarkı sözü yaz: duygu=hüzün, enerji=4, tempo=72 BPM, ton=A minor, enstrümanlar=ney, keman, piyano, vokal=erkek, kısık, dramatik'
print('GIRDI:', ornek_input)

model.eval()
inputs = tokenizer(ornek_input, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_length=MAX_TARGET_LEN,
        num_beams=4,
        no_repeat_ngram_size=3,   # tekrar engelle
        early_stopping=True,
    )

decoded = tokenizer.decode(out[0], skip_special_tokens=True)
print('\nCIKTI:')
print(decoded)

In [ ]:
# ── 9. Değerlendirme ──────────────────────────────────────────────────────────
# Metrik: [Verse] veya [Chorus] tag var mi, yeterli kelime var mi
import random

random.seed(0)
samples = random.sample(range(len(eval_ds)), min(30, len(eval_ds)))
has_tags, long_enough = 0, 0

for idx in samples:
    inp = eval_ds[idx]['input_text']
    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4,
                             no_repeat_ngram_size=3, early_stopping=True)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)

    if '[Verse' in gen or '[Chorus' in gen:
        has_tags += 1
    if len(gen.split()) >= 30:
        long_enough += 1

n = len(samples)
print(f'Yapısal tag olan : {has_tags}/{n} ({has_tags/n:.0%})')
print(f'30+ kelime olan  : {long_enough}/{n} ({long_enough/n:.0%})')

if has_tags/n >= 0.70 and long_enough/n >= 0.80:
    print('\nModel 2 hazir. Iki model birlestirilip MCP server yazilabilir.')
else:
    print('\nHenuz yeterli degil — epoch artir veya no_repeat_ngram_size dene.')

In [ ]:
# ── 10. İndir ─────────────────────────────────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/story-to-music-lyricist', 'zip', OUTPUT_DIR)
print('Output panelinden story-to-music-lyricist.zip indirebilirsin.')